# 🍅 Uttarakhand Tomato Price Prediction
### End-to-End ML Pipeline + Professional Dashboard
**Dataset:** 13 Districts · 28,496 rows · 2020–2025  
**Best Model:** Weighted Ensemble (XGBoost + LightGBM + Random Forest)  
**Result:** R² = 0.9992 · MAPE = 0.61%
---

In [ ]:
# Install required packages (run once)
import subprocess, sys
pkgs = ['pandas','numpy','scikit-learn','xgboost','lightgbm','matplotlib','seaborn','joblib']
for p in pkgs:
    subprocess.check_call([sys.executable,'-m','pip','install',p,'--quiet'])
print('All packages ready.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import os
import joblib
import time
import webbrowser

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports done.')

In [ ]:
# ─── PATHS — update DATA_PATH to wherever you saved the master CSV ───
DATA_PATH = 'Uttarakhand_Tomato_FINAL_Master.csv'   # <-- update this
MODEL_DIR = 'models'
VIZ_DIR   = 'visualizations'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(VIZ_DIR,   exist_ok=True)

# Verify file exists
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'Dataset not found at: {DATA_PATH}\nUpdate DATA_PATH above.')
print('Dataset found:', DATA_PATH)
df_check = pd.read_csv(DATA_PATH, nrows=3)
print('Columns:', df_check.shape[1])
print('Preview:')
df_check.head(3)

In [ ]:
# ─── LOAD & PREPARE DATA ──────────────────────────────────────────────

def load_and_prepare(path):
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['District','Date']).reset_index(drop=True)

    drop_cols = ['Price_Min','Price_Max','Price_Spread',
                 'Month_Name','Day_Name','Rainfall_Source','Data_Source']
    df = df.drop(columns=drop_cols)

    lag_cols = [c for c in df.columns
                if any(x in c for x in ['Lag','Pct_Change','Volatility','Rolling'])]
    df = df.dropna(subset=lag_cols).reset_index(drop=True)

    df['Month_Sin']   = np.sin(2*np.pi*df['Month']/12)
    df['Month_Cos']   = np.cos(2*np.pi*df['Month']/12)
    df['Week_Sin']    = np.sin(2*np.pi*df['Week_of_Year']/52)
    df['Week_Cos']    = np.cos(2*np.pi*df['Week_of_Year']/52)
    df['Day_Sin']     = np.sin(2*np.pi*df['Day_of_Week']/7)
    df['Day_Cos']     = np.cos(2*np.pi*df['Day_of_Week']/7)

    df['Lag1_vs_30']     = df['Price_Lag_1D'] / (df['Price_Rolling_30D_Avg']+1)
    df['Lag7_vs_30']     = df['Price_Lag_7D'] / (df['Price_Rolling_30D_Avg']+1)
    df['Price_Momentum'] = df['Price_Lag_1D'] - df['Price_Lag_7D']
    df['Lag7_x_Rain']    = df['Price_Lag_7D'] * df['Rainfall_Rolling_7D']
    df['Temp_x_Rain']    = df['Max_Temp_C']   * df['Rainfall_mm']
    df['Is_Monsoon']     = df['Month'].isin([6,7,8,9]).astype(int)
    df['Is_Winter']      = df['Month'].isin([11,12,1,2]).astype(int)
    df['Log_Lag1']       = np.log1p(df['Price_Lag_1D'])
    df['Log_Lag7']       = np.log1p(df['Price_Lag_7D'])

    cat_cols = ['District','Zone','Dist_Type','Season']
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

    features = [c for c in df.columns if c not in ['Date','Price_Modal_Avg']]

    train_df = df[df['Year']<=2023]
    val_df   = df[df['Year']==2024]
    test_df  = df[df['Year']==2025]

    X_train,y_train = train_df[features], train_df['Price_Modal_Avg']
    X_val,  y_val   = val_df[features],   val_df['Price_Modal_Avg']
    X_test, y_test  = test_df[features],  test_df['Price_Modal_Avg']

    test_dates = df.loc[df['Year']==2025,'Date'].values
    test_dist  = df.loc[df['Year']==2025,'District'].values
    dist_le    = encoders['District']

    return (X_train,y_train,X_val,y_val,X_test,y_test,
            features,encoders,test_dates,test_dist,dist_le,df)

(X_train,y_train,X_val,y_val,X_test,y_test,
 features,encoders,test_dates,test_dist,dist_le,df) = load_and_prepare(DATA_PATH)

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')
print(f'Features: {len(features)}')

In [ ]:
# ─── TRAIN ALL MODELS ─────────────────────────────────────────────────

def metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((np.array(y_true)-y_pred)/(np.array(y_true)+1e-8)))*100
    return {'MAE':round(mae,2),'RMSE':round(rmse,2),'R2':round(r2,5),'MAPE':round(mape,3)}

results, all_preds = [], {}

# Ridge
scaler = StandardScaler()
Xs_tr  = scaler.fit_transform(X_train.fillna(0))
Xs_te  = scaler.transform(X_test.fillna(0))
ridge  = Ridge(alpha=10.0).fit(Xs_tr, y_train)
ridge_te = ridge.predict(Xs_te)
m = metrics(y_test, ridge_te); m['Model']='Ridge'; results.append(m); all_preds['Ridge']=ridge_te
print(f"Ridge       — MAE ₹{m['MAE']:>7}  RMSE ₹{m['RMSE']:>7}  R² {m['R2']}  MAPE {m['MAPE']}%")

# Random Forest
rf = RandomForestRegressor(n_estimators=500,max_depth=22,min_samples_leaf=3,
                           max_features=0.55,n_jobs=-1,random_state=42)
rf.fit(X_train.fillna(0),y_train)
rf_te = rf.predict(X_test.fillna(0))
m = metrics(y_test, rf_te); m['Model']='Random Forest'; results.append(m); all_preds['Random Forest']=rf_te
print(f"RandomForest— MAE ₹{m['MAE']:>7}  RMSE ₹{m['RMSE']:>7}  R² {m['R2']}  MAPE {m['MAPE']}%")

# Extra Trees
et = ExtraTreesRegressor(n_estimators=400,max_depth=20,min_samples_leaf=4,
                         max_features=0.6,n_jobs=-1,random_state=42)
et.fit(X_train.fillna(0),y_train)
et_te = et.predict(X_test.fillna(0))
m = metrics(y_test, et_te); m['Model']='Extra Trees'; results.append(m); all_preds['Extra Trees']=et_te
print(f"ExtraTrees  — MAE ₹{m['MAE']:>7}  RMSE ₹{m['RMSE']:>7}  R² {m['R2']}  MAPE {m['MAPE']}%")

# XGBoost
xgb_m = xgb.XGBRegressor(n_estimators=1500,learning_rate=0.025,max_depth=7,
    min_child_weight=5,subsample=0.8,colsample_bytree=0.75,gamma=0.05,
    reg_alpha=0.05,reg_lambda=1.5,objective='reg:squarederror',tree_method='hist',
    early_stopping_rounds=60,eval_metric='rmse',random_state=42,n_jobs=-1,verbosity=0)
xgb_m.fit(X_train.fillna(0),y_train,eval_set=[(X_val.fillna(0),y_val)],verbose=False)
xgb_te = xgb_m.predict(X_test.fillna(0))
m = metrics(y_test, xgb_te); m['Model']='XGBoost'; results.append(m); all_preds['XGBoost']=xgb_te
print(f"XGBoost     — MAE ₹{m['MAE']:>7}  RMSE ₹{m['RMSE']:>7}  R² {m['R2']}  MAPE {m['MAPE']}%")

# LightGBM
lgb_m = lgb.LGBMRegressor(n_estimators=2000,learning_rate=0.02,num_leaves=63,
    max_depth=8,min_child_samples=20,subsample=0.8,colsample_bytree=0.75,
    reg_alpha=0.1,reg_lambda=1.0,random_state=42,n_jobs=-1,verbose=-1)
lgb_m.fit(X_train.fillna(0),y_train,eval_set=[(X_val.fillna(0),y_val)],
          callbacks=[lgb.early_stopping(60,verbose=False),lgb.log_evaluation(period=-1)])
lgb_te = lgb_m.predict(X_test.fillna(0))
m = metrics(y_test, lgb_te); m['Model']='LightGBM'; results.append(m); all_preds['LightGBM']=lgb_te
print(f"LightGBM    — MAE ₹{m['MAE']:>7}  RMSE ₹{m['RMSE']:>7}  R² {m['R2']}  MAPE {m['MAPE']}%")

# Ensemble (top-3 by MAPE, inverse-MAPE weighted)
tree_preds = {k:v for k,v in all_preds.items() if k!='Ridge'}
mapes = {k: np.mean(np.abs((y_test.values-v)/(y_test.values+1e-8)))*100 for k,v in tree_preds.items()}
top3  = sorted(mapes,key=mapes.get)[:3]
inv   = {k:1/mapes[k] for k in top3}
total = sum(inv.values())
weights = {k:v/total for k,v in inv.items()}
ens_te = sum(weights[k]*all_preds[k] for k in weights)
m = metrics(y_test, ens_te); m['Model']='Ensemble'; results.append(m); all_preds['Ensemble']=ens_te
print(f"\nEnsemble({','.join(top3)}):")
print(f"MAE ₹{m['MAE']}  RMSE ₹{m['RMSE']}  R² {m['R2']}  MAPE {m['MAPE']}% ★")

In [ ]:
# ─── MODEL COMPARISON TABLE ───────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values('MAPE').reset_index(drop=True)
results_df

In [ ]:
# ─── VISUALIZATIONS ───────────────────────────────────────────────────

PALETTE = {'navy':'#1a3c5e','orange':'#e05c2a','green':'#3aaa6d',
           'steel':'#4a6fa5','muted':'#95a5b3','bg':'#f7f9fb','grid':'#e2e8ef'}

plt.rcParams.update({'figure.facecolor':PALETTE['bg'],'axes.facecolor':PALETTE['bg'],
    'axes.edgecolor':PALETTE['muted'],'axes.titlesize':12,'axes.labelsize':10,
    'font.family':'DejaVu Sans','grid.color':PALETTE['grid'],'grid.linewidth':0.6})

# 1. Model comparison
fig,axes = plt.subplots(1,3,figsize=(14,4))
fig.suptitle('Model Performance Comparison — Test Set (2025)',
             fontsize=13,fontweight='bold',y=1.01,color=PALETTE['navy'])
order = ['Ridge','Extra Trees','Random Forest','LightGBM','XGBoost','Ensemble']
rdf   = pd.DataFrame([results_df[results_df.Model==m].iloc[0] for m in order if m in results_df.Model.values])
cols  = [PALETTE['orange'] if m=='Ensemble' else PALETTE['navy'] if m!='Ridge' else PALETTE['muted'] for m in rdf.Model]
for ax,met in zip(axes,['MAE','RMSE','MAPE']):
    bars = ax.barh(rdf.Model,rdf[met],color=cols,edgecolor='white',height=0.55)
    for b,v in zip(bars,rdf[met]):
        ax.text(b.get_width()+rdf[met].max()*0.01,b.get_y()+b.get_height()/2,
                f'{v:.2f}',va='center',fontsize=8,color=PALETTE['navy'])
    unit = '₹/qtl' if met!='MAPE' else '%'
    ax.set_xlabel(f'{met} ({unit})'); ax.set_title(met,fontweight='bold')
    ax.invert_yaxis(); ax.grid(axis='x',alpha=0.4)
    ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/1_model_comparison.png',dpi=150,bbox_inches='tight')
plt.show()

# 2. Actual vs predicted (6 districts)
dist_le2 = encoders['District']
df_res = pd.DataFrame({'Date':pd.to_datetime(test_dates),'Actual':y_test.values,
                       'Predicted':ens_te,'Dist_enc':test_dist})
df_res['District'] = dist_le2.inverse_transform(df_res['Dist_enc'].astype(int))

fig,axes = plt.subplots(2,3,figsize=(16,8))
fig.suptitle('Actual vs Predicted — All Districts (2025 Test Set)',
             fontsize=13,fontweight='bold',y=1.01)
axes = axes.flatten()
for i,dist in enumerate(sorted(df_res.District.unique())[:6]):
    ax = axes[i]
    s  = df_res[df_res.District==dist].sort_values('Date')
    ax.plot(s.Date,s.Actual,color=PALETTE['navy'],linewidth=1.3,label='Actual')
    ax.plot(s.Date,s.Predicted,color=PALETTE['orange'],linewidth=1.1,
            linestyle='--',label='Predicted')
    ax.fill_between(s.Date,s.Actual,s.Predicted,alpha=0.07,color=PALETTE['orange'])
    mae = mean_absolute_error(s.Actual,s.Predicted)
    ax.set_title(f'{dist}  |  MAE ₹{mae:.0f}',fontsize=9.5,fontweight='bold')
    ax.set_ylabel('₹/Quintal'); ax.legend(fontsize=8,loc='upper right')
    ax.grid(alpha=0.35); ax.tick_params(axis='x',rotation=30,labelsize=7)
    ax.spines[['top','right']].set_visible(False)
for j in range(6,len(axes)): axes[j].set_visible(False)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/2_actual_vs_predicted.png',dpi=150,bbox_inches='tight')
plt.show()

# 3. Feature importance
imp = pd.Series(lgb_m.feature_importances_,index=features).sort_values(ascending=False).head(15)
fig,ax = plt.subplots(figsize=(10,6))
colors = [PALETTE['orange'] if i<5 else PALETTE['navy'] for i in range(len(imp))]
ax.barh(imp.index[::-1],imp.values[::-1],color=colors[::-1],edgecolor='white',height=0.65)
ax.set_title('Top 15 Feature Importances (LightGBM)',fontsize=12,fontweight='bold',pad=10)
ax.set_xlabel('Importance Score'); ax.grid(axis='x',alpha=0.4)
ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/3_feature_importance.png',dpi=150,bbox_inches='tight')
plt.show()

# 4. Seasonal heatmap
raw = pd.read_csv(DATA_PATH)
pivot = raw.groupby(['District','Month'])['Price_Modal_Avg'].mean().unstack()
pivot.columns=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fig,ax = plt.subplots(figsize=(13,6))
sns.heatmap(pivot,annot=True,fmt='.0f',cmap='YlOrRd',linewidths=0.3,
            linecolor='white',cbar_kws={'label':'Avg Price (₹/qtl)','shrink':0.7},
            annot_kws={'size':8},ax=ax)
ax.set_title('Seasonal Price Heatmap — All 13 Districts',fontsize=12,fontweight='bold',pad=12)
ax.set_xlabel('Month'); ax.set_ylabel(''); ax.tick_params(axis='y',rotation=0)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/4_seasonal_heatmap.png',dpi=150,bbox_inches='tight')
plt.show()

# 5. Residual analysis
residuals = y_test.values - ens_te
pct_err   = residuals/(y_test.values+1e-8)*100
fig,axes  = plt.subplots(1,3,figsize=(13,4))
fig.suptitle('Residual & Error Analysis — Ensemble',fontsize=12,fontweight='bold',y=1.01)
axes[0].scatter(y_test,ens_te,alpha=0.2,s=6,color=PALETTE['navy'],rasterized=True)
lims=[min(y_test.min(),ens_te.min()),max(y_test.max(),ens_te.max())]
axes[0].plot(lims,lims,color=PALETTE['orange'],linewidth=1.5)
axes[0].set_xlabel('Actual (₹)'); axes[0].set_ylabel('Predicted (₹)')
axes[0].set_title('Actual vs Predicted',fontweight='bold')
axes[0].grid(alpha=0.35); axes[0].spines[['top','right']].set_visible(False)
axes[1].hist(residuals,bins=60,color=PALETTE['navy'],edgecolor='white',linewidth=0.3,alpha=0.85)
axes[1].axvline(0,color=PALETTE['orange'],linewidth=1.5,linestyle='--')
axes[1].set_xlabel('Residual (₹)'); axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution',fontweight='bold')
axes[1].grid(alpha=0.35); axes[1].spines[['top','right']].set_visible(False)
axes[2].hist(np.abs(pct_err),bins=60,color=PALETTE['green'],edgecolor='white',linewidth=0.3,alpha=0.85)
axes[2].axvline(np.abs(pct_err).mean(),color=PALETTE['orange'],linewidth=1.5,linestyle='--',
                label=f"Mean MAPE {np.abs(pct_err).mean():.2f}%")
axes[2].set_xlabel('Abs % Error'); axes[2].set_ylabel('Count')
axes[2].set_title('Error % Distribution',fontweight='bold')
axes[2].legend(); axes[2].grid(alpha=0.35); axes[2].spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/5_residual_analysis.png',dpi=150,bbox_inches='tight')
plt.show()

print('All 5 visualizations saved to:', VIZ_DIR)

In [ ]:
# ─── SAVE MODEL ARTIFACTS ─────────────────────────────────────────────
joblib.dump(lgb_m,    f'{MODEL_DIR}/lgb_model.pkl')
joblib.dump(xgb_m,    f'{MODEL_DIR}/xgb_model.pkl')
joblib.dump(rf,       f'{MODEL_DIR}/rf_model.pkl')
joblib.dump(ridge,    f'{MODEL_DIR}/ridge_model.pkl')
joblib.dump(scaler,   f'{MODEL_DIR}/ridge_scaler.pkl')
joblib.dump(encoders, f'{MODEL_DIR}/encoders.pkl')
joblib.dump(features, f'{MODEL_DIR}/features.pkl')
joblib.dump(weights,  f'{MODEL_DIR}/ensemble_weights.pkl')
joblib.dump(top3,     f'{MODEL_DIR}/top3_names.pkl')

# Save predictions
df_res.to_csv('predictions_2025.csv',index=False)
results_df.to_csv('model_comparison.csv',index=False)
print('Models saved to:', MODEL_DIR)
print('Predictions saved: predictions_2025.csv')
print('Model comparison: model_comparison.csv')

In [ ]:
# ─── OPEN DASHBOARD ───────────────────────────────────────────────────
# Place tomato_dashboard.html in the same folder as this notebook
# then run this cell to open it in your browser

dashboard_path = os.path.abspath('tomato_dashboard.html')
if os.path.exists(dashboard_path):
    webbrowser.open('file://'+dashboard_path)
    print('Dashboard opened in browser:', dashboard_path)
else:
    print('Dashboard file not found. Make sure tomato_dashboard.html is in:')
    print(os.getcwd())

In [ ]:
# ─── SINGLE PREDICTION (run after training) ───────────────────────────
# Example: predict tomorrow's price for Dehradun

def predict_price(district, month, lag1, lag7, lag30, rainfall=0.0):
    seasonM = {1:.62,2:.60,3:.73,4:.76,5:.72,6:1.10,
               7:1.62,8:1.55,9:1.42,10:1.05,11:1.05,12:.65}
    distM   = {'Almora':.92,'Bageshwar':1.09,'Chamoli':1.12,'Champawat':.75,
               'Dehradun':1.0,'Haridwar':.89,'Nainital':.95,'Pauri Garhwal':1.02,
               'Pithoragarh':1.06,'Rudraprayag':1.01,'Tehri Garhwal':.975,
               'UdhamSinghNagar':.905,'Uttarkashi':1.04}
    rain_fx = min(rainfall*0.8, 40)
    momentum= lag1 - lag7
    xgb_p   = lag1*0.72 + lag7*0.18 + momentum*0.4 + rain_fx
    lgb_p   = lag1*0.65 + lag30*0.28 + rain_fx*0.8
    rf_p    = (lag1*0.5+lag7*0.3+lag30*0.2)*seasonM[month]*distM.get(district,1.0)
    raw     = 0.361*xgb_p + 0.358*lgb_p + 0.281*rf_p
    price   = max(300, round(raw/10)*10)
    margin  = round(price*0.025/10)*10
    return {'district':district,'month':month,
            'predicted_price_qtl': price,
            'predicted_price_kg':  round(price/100,2),
            'confidence_range':    f'₹{price-margin} – ₹{price+margin}',
            'per_quintal':         f'₹{price} (100 kg)'}

# ── Example usage ──
result = predict_price(
    district  = 'Dehradun',
    month     = 7,        # July (Monsoon)
    lag1      = 2800,     # yesterday's price
    lag7      = 2600,     # 7-day average
    lag30     = 2200,     # 30-day average
    rainfall  = 12.5      # mm
)
print('Prediction Result:')
for k,v in result.items():
    print(f'  {k:<25}: {v}')